<a href="https://colab.research.google.com/github/alveslim/AI-Engineering/blob/main/Treino_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade pip
!pip install xformers
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.6 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 11.4 MB/s  0:00:00
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ybaiibtc/unsloth_c6c9aa5656884beabc40570bdd2d299d
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ybaiibtc/unsloth_c6c9aa5656884beabc40570bdd2d299d
  Resolved https://github.com/unslothai/unsloth.git to commit 364a42f02fd865cdf2168e755587ace4869f3fd4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.29.1
    

In [ ]:
from unsloth import FastLanguageModel
import torch
import pandas as pd
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. CARREGAR O MODELO
print("A carregar o modelo...")
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 2. PREPARAR OS DADOS
print("A preparar o dataset...")
dados = {
    "instruction": [
        "És um assistente de controlo de qualidade de produção.",
        "És um assistente de controlo de qualidade de produção.",
        "És um engenheiro de software especialista em Python."
    ],
    "input": [
        "Qual é o procedimento se detetar uma falha na colagem das caixas?",
        "Como registo um lote reprovado no sistema de OPs?",
        "Qual é a melhor forma de exportar relatórios de produção para PDF?"
    ],
    "output": [
        "Deve separar imediatamente o lote afetado, colocar a etiqueta vermelha de 'Quarentena' e reportar ao supervisor de turno para recalibragem da máquina.",
        "Abra o Gerador de OPs, selecione a aba 'Relatórios', insira o número do lote e marque a opção 'Desvio de Qualidade' antes de submeter e gerar o CSV.",
        "Recomendo a utilização da biblioteca ReportLab em conjunto com o Pandas. O Pandas trata a manipulação e limpeza dos dados, e o ReportLab desenha o layout do documento."
    ]
}

dataset = Dataset.from_pandas(pd.DataFrame(dados))

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions, inputs, outputs = examples["instruction"], examples["input"], examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        texts.append(alpaca_prompt.format(instruction, input, output) + EOS_TOKEN)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

# 3. INICIAR O TREINO
print("A iniciar o treino (Fine-Tuning)...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print("Treino concluído com sucesso!")

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1606: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
A carregar o modelo...
==((====))==  Unsloth 2026.9.14: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.14 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


A preparar o dataset...


...

Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/3 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/

Step,Training Loss
1,2.785929
2,2.785929
3,2.720787
4,2.494527
5,2.192529
6,1.829932
7,1.414288
8,0.992941
9,0.647811
10,0.396057


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


Treino concluído com sucesso!


In [ ]:
# Prepara o modelo para aceleração nativa de inferência (2x mais rápido)
FastLanguageModel.for_inference(model)

# Criar a pergunta no mesmo formato que usamos no treino (Alpaca)
inputs = tokenizer(
[
    alpaca_prompt.format(
        "És um assistente de controlo de qualidade de produção.", # Instrução
        "Como registo um lote reprovado no sistema de OPs?", # Pergunta
        "", # A resposta fica em branco para a IA gerar
    )
], return_tensors = "pt").to("cuda")

# Gerar a resposta
outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
resposta = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print("\n--- RESPOSTA DA IA ---")
print(resposta)

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



--- RESPOSTA DA IA ---
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
És um assistente de controlo de qualidade de produção.

### Input:
Como registo um lote reprovado no sistema de OPs?

### Response:
Abra o Gerador de OPs, selecione a aba 'Relatórios', insira o número do lote e marque a opção 'Desvio de Qualidade' antes de submeter e gerar o CSV.


In [ ]:
print("A iniciar a conversão para GGUF (isto pode demorar uns 5 a 10 minutos)...")

# O método q4_k_m comprime o modelo para cerca de 4.8GB mantendo uma elevada inteligência
model.save_pretrained_gguf("modelo_qualidade_gguf", tokenizer, quantization_method = "q4_k_m")

print("Conversão concluída com sucesso!")

A iniciar a conversão para GGUF (isto pode demorar uns 5 a 10 minutos)...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in modelo_qualidade_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00004-of-00004.safetensors: downloading bytes:           |  0.00B            



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:47<00:00, 11.94s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:49<00:00, 12.43s/it]


Unsloth: Merge process complete. Saved to `/content/modelo_qualidade_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11160-mix-a6922cc (app-b11160-mix-a6922cc-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['modelo_qualidade_gguf_gguf/llama-3-8b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions complet

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Define o tamanho máximo de contexto/texto
dtype = None
load_in_4bit = True # Essencial para não esgotar a RAM da GPU

# 1. Carregar o modelo e o "tradutor" de texto (tokenizer)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Adicionar os adaptadores LoRA (esta é a parte do cérebro que vai efetivamente ser treinada com os seus dados)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("Modelo carregado e pronto para treino!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.14: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.14 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Modelo carregado e pronto para treino!


In [ ]:
import pandas as pd
from datasets import Dataset

# 1. Criar os nossos dados de treino usando o Pandas
dados = {
    "instruction": [
        "És um assistente de controlo de qualidade de produção.",
        "És um assistente de controlo de qualidade de produção.",
        "És um engenheiro de software especialista em Python."
    ],
    "input": [
        "Qual é o procedimento se detetar uma falha na colagem das caixas?",
        "Como registo um lote reprovado no sistema de OPs?",
        "Qual é a melhor forma de exportar relatórios de produção para PDF?"
    ],
    "output": [
        "Deve separar imediatamente o lote afetado, colocar a etiqueta vermelha de 'Quarentena' e reportar ao supervisor de turno para recalibragem da máquina.",
        "Abra o Gerador de OPs, selecione a aba 'Relatórios', insira o número do lote e marque a opção 'Desvio de Qualidade' antes de submeter e gerar o CSV.",
        "Recomendo a utilização da biblioteca ReportLab em conjunto com o Pandas. O Pandas trata a manipulação e limpeza dos dados, e o ReportLab desenha o layout do documento."
    ]
}

# Converter para DataFrame e depois para o formato do Hugging Face
df = pd.DataFrame(dados)
dataset = Dataset.from_pandas(df)

# 2. Definir o formato que o modelo entende (Alpaca Prompt)
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Token que avisa a IA que a resposta terminou

# 3. Função para juntar tudo no formato final
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Substitui as chavetas {} pelos nossos dados e adiciona o token de fim
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Aplicar a formatação ao dataset
dataset = dataset.map(formatting_prompts_func, batched = True)

print("Dataset preparado com sucesso!")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset preparado com sucesso!


In [ ]:
!pip install "datasets<5.0.0" "transformers<=5.17.0" "trl<=1.13.0"

  Using cached trl-1.13.0-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 16.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 124.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.7 MB/s  0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 5.0.1
    Uninstalling datasets-5.0.1:
      Successfully uninstalled datasets-5.0.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0
  Attempting uninstall: trl
    Found existing installation: trl 1.14.1
    Uninstalling trl-1.14.1:
      Successfully uninstalled trl-1.14.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [trl]


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("A iniciar o treino (Fine-Tuning)...")
trainer_stats = trainer.train()
print("Treino concluído com sucesso!")

NameError: name 'model' is not defined